# 04. 演習1 — 報酬関数の改善実験

**対応するテキスト**: [docs/07_演習1_報酬関数の改善.md](../docs/07_演習1_報酬関数の改善.md)

**変えるのは `--reward-mode` と `--shaping-weight` だけ**です。他はベースラインと同一にします
（**一度に 1 要素だけ変える**原則）。

> ⚠ **実行前に、[docs/07](../docs/07_演習1_報酬関数の改善.md) の 7.3 で仮説を書いてください。**
> 結果を見てから解釈を作ると「後付けの物語」になり、実験の価値が失われます。

In [ ]:
from azure.ai.ml import MLClient, command
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<SUBSCRIPTION_ID>"
RESOURCE_GROUP = "<RESOURCE_GROUP>"
WORKSPACE_NAME = "<AML_WORKSPACE_NAME>"

COMPUTE_NAME = "cpu-cluster"
ENV_REF = "rl-panda-gym-env@latest"
EXPERIMENT = "rl-reward-exp"

TAGS = {
    "project": "rl-workshop",
    "owner": "<your-alias>",
    "delete-after": "<YYYY-MM-DD>",
    "phase": "reward",
}

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
ws = ml_client.workspaces.get(WORKSPACE_NAME)
print("接続しました:", ws.name)

## 1. ベースライン条件（ベースライン実験と同一）

> **この値はベースライン実験と 1 文字も変えません。** 変えると比較が成立しません。

In [ ]:
BASELINE = dict(
    algo="sac",
    total_timesteps=50_000,
    seed=0,
    learning_rate=3e-4,
    gamma=0.99,
    batch_size=256,
    buffer_size=200_000,
    learning_starts=1_000,
    eval_freq=5_000,
    n_eval_episodes=30,
    final_eval_episodes=100,
)

ENV_ID = "PandaPickAndPlace-v3"
USE_HER = 1


def submit_reward_exp(reward_mode: str, shaping_weight: float,
                      reward_fn_version: str, display_name: str):
    cfg = dict(BASELINE)
    cfg["reward_mode"] = reward_mode
    cfg["shaping_weight"] = shaping_weight
    cfg["reward_fn_version"] = reward_fn_version

    parts = ["python train_rl.py", f"--env-id {ENV_ID}", f"--use-her {USE_HER}"]
    for key, value in cfg.items():
        parts.append(f"--{key.replace('_', '-')} {value}")

    job = command(
        code="../src",
        command=" ".join(parts),
        environment=ENV_REF,
        compute=COMPUTE_NAME,
        experiment_name=EXPERIMENT,
        display_name=display_name,
        tags={**TAGS, "reward_mode": reward_mode, "shaping_weight": str(shaping_weight)},
    )
    returned = ml_client.jobs.create_or_update(job)
    print(f"投入: {display_name:38s} -> {returned.name}")
    return returned

## 2. 4 条件を投入する

| 条件 | 報酬 | 想定される副作用 |
|---|---|---|
| `sparse` | 成功 0 / 失敗 -1 | 学習がほぼ進まない |
| `dense` | -（距離） | **押すだけで距離が縮むので、押す行動が強化されやすい** |
| `sparse_shaped` | sparse - w×距離 | w が大きいと dense と同じ副作用 |
| `sparse_time_penalty` | sparse - w（毎ステップ） | **早く失敗して終わらせる方向に働く危険** |

> ⚠ **クォータに注意。** 4 本同時はノードを 4 つ使います。足りなければ 2 本ずつに分けてください。

In [ ]:
reward_jobs = {}

reward_jobs["sparse"] = submit_reward_exp(
    "sparse", 0.0, "v1", "reward_pnp_sparse_seed0_v1")

reward_jobs["dense"] = submit_reward_exp(
    "dense", 0.0, "v2", "reward_pnp_dense_seed0_v2")

reward_jobs["shaped_0.1"] = submit_reward_exp(
    "sparse_shaped", 0.1, "v2", "reward_pnp_shaped-w0.1_seed0_v2")

reward_jobs["time_penalty_0.02"] = submit_reward_exp(
    "sparse_time_penalty", 0.02, "v2", "reward_pnp_timepen-w0.02_seed0_v2")

print("\n投入した 4 本:")
for k, v in reward_jobs.items():
    print(f"  {k:20s} {v.name}")
    print(f"  {'':20s} {v.studio_url}")

In [ ]:
# 進捗確認
for key, job in reward_jobs.items():
    print(f"{key:20s} {ml_client.jobs.get(job.name).status}")

## 3. 比較表を作る

> **重要**: 評価は **すべての条件で `sparse` に固定**されています（`eval_reward_mode` パラメーターで確認できます）。
> そのため `final_mean_reward` を条件間で横並び比較できます。
> 詳しくは [docs/07](../docs/07_演習1_報酬関数の改善.md) の 7.4 を参照してください。

In [ ]:
import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

try:
    tracking_uri = ml_client.workspaces.get(WORKSPACE_NAME).mlflow_tracking_uri
except AttributeError:
    tracking_uri = (
        f"azureml://{ws.location}.api.azureml.ms/mlflow/v1.0"
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
    )
mlflow.set_tracking_uri(tracking_uri)
client = MlflowClient()

METRICS = [
    "final_success_rate",
    "final_success_rate_stderr",
    "final_mean_reward",
    "final_std_reward",
    "final_mean_episode_length",
    "train_minutes",
]
PARAMS = ["reward_mode", "shaping_weight", "eval_reward_mode",
          "reward_fn_version", "her_effective", "seed"]

rows = []
for key, job in reward_jobs.items():
    run = mlflow.get_run(job.name)
    row = {"条件": key}
    row.update({p: run.data.params.get(p) for p in PARAMS})
    row.update({m: run.data.metrics.get(m) for m in METRICS})
    row["動画の観察"] = ""   # ← 手で埋めてください
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("reward_comparison.csv", index=False, encoding="utf-8-sig")
df

## 4. 差が「誤差の範囲」でないかを確認する

> ⚠ **成功率 0.30 と 0.33 の差は、100 エピソード評価では誤差の範囲です。**
> 必ず標準誤差と比べてください（[docs/06](../docs/06_結果の読み解き.md) 6.4）。

In [ ]:
sparse_rows = df[df["条件"] == "sparse"]

if len(sparse_rows) == 0 or sparse_rows.iloc[0]["final_success_rate"] is None:
    print("[ERROR] 基準となる sparse 条件の結果が見つかりません。")
    print("        4 本のジョブがすべて Completed になっているか確認してください。")
else:
    base = sparse_rows.iloc[0]
    print(f"基準 (sparse): 成功率 = {base['final_success_rate']:.3f} "
          f"± {base['final_success_rate_stderr']:.3f}\n")

    for _, row in df.iterrows():
        if row["条件"] == "sparse":
            continue
        if row["final_success_rate"] is None:
            print(f"{row['条件']:20s} 結果なし（ジョブ未完了）")
            continue
        diff = row["final_success_rate"] - base["final_success_rate"]
        # 2 条件の差の標準誤差 = sqrt(se1^2 + se2^2)
        se = (row["final_success_rate_stderr"] ** 2
              + base["final_success_rate_stderr"] ** 2) ** 0.5
        # 差が「差の標準誤差の 2 倍」を超えていれば、偶然では説明しにくい
        verdict = "差あり(誤差の2倍超)" if abs(diff) > 2 * se else "誤差の範囲(差は言えない)"
        print(f"{row['条件']:20s} 差 = {diff:+.3f}  (差の標準誤差 {se:.3f})  → {verdict}")

## 5. 学習曲線を重ねて比較する

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for metric, ax in zip(["eval_success_rate", "eval_mean_reward"], axes):
    for key, job in reward_jobs.items():
        try:
            hist = sorted(client.get_metric_history(job.name, metric), key=lambda m: m.step)
        except Exception as exc:
            print(f"[WARN] {key}/{metric}: {exc}")
            continue
        if hist:
            ax.plot([m.step for m in hist], [m.value for m in hist], marker="o", label=key)
    ax.set_xlabel("timesteps")
    ax.set_title(metric)
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 6. 【必須】4 条件すべての動画を見る

> ⚠ **数値だけで判断しないでください。**
> 特に **`dense` で成功率が 30% 付近に張り付いている場合**は、
> 「キューブを押して転がしているだけ」の可能性があります
> （[docs/06](../docs/06_結果の読み解き.md) 6.3）。
>
> **動画でグリッパーが閉じているかを確認してください。**

In [ ]:
import os

os.makedirs("downloads", exist_ok=True)
for key, job in reward_jobs.items():
    artifacts = [a.path for a in client.list_artifacts(job.name)]
    if "eval_video.mp4" in artifacts:
        local = client.download_artifacts(
            run_id=job.name, path="eval_video.mp4", dst_path=f"downloads/{key}"
        )
        print(f"{key:20s} -> {local}")
    else:
        print(f"{key:20s} -> eval_video.mp4 なし（docs/05 の TS #9 参照）")

## 7. ✅ チェックリスト

- [ ] **実行前に**仮説（期待する結果と副作用）を書いた
- [ ] 4 条件がすべて `Completed` になった
- [ ] `eval_reward_mode` が **4 条件すべて `sparse`** であることを確認した
- [ ] 比較表（`reward_comparison.csv`）を作成した
- [ ] **差が誤差の範囲かどうかを判定した**（4 節）
- [ ] **4 条件すべての動画を再生した**
- [ ] 「動画の観察」欄を埋めた
- [ ] **効果を確認できなかった条件も記録した**